# DEKM 2021 deterministic reproduction check: REUTERS seed 42

This notebook runs seed 42 twice in isolated output directories. The run is accepted only when both manifests report identical metrics and output hashes.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a GPU accelerator in Kaggle notebook settings."

In [ ]:
repo_url = "https://github.com/vdat-18/mvdec_fpmax.git"
repo_dir = "/kaggle/working/mvdec_fpmax"
!rm -rf {repo_dir}
!git clone --branch dev --single-branch {repo_url} {repo_dir}
%cd {repo_dir}
!python -m pip install -q -e . gdown

In [ ]:
!python scripts/fetch_dekm_release.py --datasets REUTERS

for output_dir in [
    '/kaggle/working/dekm-repeat-a',
    '/kaggle/working/dekm-repeat-b',
]:
    !dekm-reproduce REUTERS \
        --seed 42 --runs 1 --require-gpu \
        --dataset-root external_repos/DEKM/datasets \
        --output-dir {output_dir}


In [ ]:
import json
from pathlib import Path


def load_manifest(output_dir):
    root = Path(output_dir) / 'reuters' / 'dekm_2021_released_code_v1'
    path = next(root.glob('*/seed_42/manifest.json'))
    return json.loads(path.read_text())


first = load_manifest('/kaggle/working/dekm-repeat-a')
second = load_manifest('/kaggle/working/dekm-repeat-b')
assert first['runtime']['tensorflow_op_determinism'] is True
assert second['runtime']['tensorflow_op_determinism'] is True
assert first['metrics'] == second['metrics']
assert first['output_sha256'] == second['output_sha256']
print('Deterministic replay passed')
print(json.dumps(first['metrics'], indent=2))
print(json.dumps(first['output_sha256'], indent=2))


In [ ]:
!cd /kaggle/working && zip -qr dekm-reuters-seed42-repeatability.zip dekm-repeat-a dekm-repeat-b
print('Result: /kaggle/working/dekm-reuters-seed42-repeatability.zip')
